In [1]:
# Google Colab specific imports
from google.colab import drive
# Core libraries
import numpy as np
import pandas as pd
import os
import time
import shutil

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Visualization
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
sns.set_style('darkgrid')

# Machine Learning
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras

# Enable mixed precision for faster training on GPU
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('float32')
mixed_precision.set_global_policy(policy)
print('Compute dtype: %s' % policy.compute_dtype)
print('Variable dtype: %s' % policy.variable_dtype)
# Keras modules
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Activation, Dropout, Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras import regularizers
from tensorflow.keras.models import Model

# Check GPU availability
print("GPU Available:", tf.config.list_physical_devices('GPU'))
if tf.config.list_physical_devices('GPU'):
    print("GPU detected! Training will be fast.")
else:
    print("No GPU detected. Training will be slow.")

Compute dtype: float32
Variable dtype: float32
GPU Available: []
No GPU detected. Training will be slow.


In [2]:
# Mount Google Drive
drive.mount('/content/drive')

start_time=time.time()
sdir = '/content/drive/MyDrive/dataset_train'
working_dir = '/content/cauliflower_training/'
img_height = 300
img_width = 300

# Create working directory first
os.makedirs(working_dir, exist_ok=True)

save_path = os.path.join(working_dir, 'cali')

if os.path.isdir(save_path):
    shutil.rmtree(save_path) # if the directory exists remove it

os.makedirs(save_path, exist_ok=True)  # Changed from os.mkdir() to os.makedirs()

classlist = sorted(os.listdir(sdir))
for klass in classlist:
    classpath = os.path.join(sdir, klass)
    if not os.path.isdir(classpath):  # skip if not a directory
        continue
    print('processing ', klass,  ' class')
    class_dest_path = os.path.join(save_path, klass)
    os.makedirs(class_dest_path, exist_ok=True)  # Changed here too
    flist = os.listdir(classpath)
    for f in flist:
        fpath = os.path.join(classpath,f)
        f_dest_path = os.path.join(class_dest_path, f)
        try: # make sure images are correct image files
            img = cv2.imread(fpath)
            img = cv2.resize(img, (img_height, img_width))
            cv2.imwrite(f_dest_path, img)
        except:
            print('Image ', fpath, ' is an invalid image file')

tr_duration = time.time() - start_time
hours = tr_duration // 3600
minutes = (tr_duration - (hours * 3600)) // 60
seconds = tr_duration - ((hours * 3600) + (minutes * 60))
msg = f'resizing elapsed time was {str(hours)} hours, {minutes:4.1f} minutes, {seconds:4.2f} seconds)'
print(msg, flush=True)

Mounted at /content/drive
processing  Alternaria Leaf Spot  class
processing  Bacterial Soft Rot  class
processing  Black Rot  class
processing  Downy Mildew  class
processing  Healthy  class
resizing elapsed time was 0.0 hours, 10.0 minutes, 29.33 seconds)


In [3]:
def make_df(save_path):
    classlist=os.listdir(save_path)
    filepaths=[]
    labels=[]
    for klass in classlist:
        classpath=os.path.join(save_path, klass)
        flist=os.listdir(classpath)
        for f in flist:
            fpath=os.path.join(classpath,f)
            filepaths.append(fpath)
            labels.append(klass)
    Fseries=pd.Series(filepaths, name='filepaths')
    Lseries=pd.Series(labels, name='labels')
    df=pd.concat([Fseries, Lseries], axis=1)
    train_df, dummy_df=train_test_split(df, train_size=.8, shuffle=True, random_state=123, stratify=df['labels'])
    valid_df, test_df=train_test_split(dummy_df,train_size=.5, shuffle=True, random_state=123, stratify=dummy_df['labels'])
    print('train_df lenght: ', len(train_df), '  test_df length: ', len(test_df), '  valid_df length: ', len(valid_df))
    # get the number of classes and the images count for each class in train_df
    classes=sorted(list(train_df['labels'].unique()))
    class_count = len(classes)
    print('The number of classes in the dataset is: ', class_count)
    groups=train_df.groupby('labels')
    print('{0:^30s} {1:^13s}'.format('CLASS', 'IMAGE COUNT'))
    countlist=[]
    classlist=[]
    for label in sorted(list(train_df['labels'].unique())):
        group=groups.get_group(label)
        countlist.append(len(group))
        classlist.append(label)
        print('{0:^30s} {1:^13s}'.format(label, str(len(group))))

    # get the classes with the minimum and maximum number of train images
    max_value=np.max(countlist)
    max_index=countlist.index(max_value)
    max_class=classlist[max_index]
    min_value=np.min(countlist)
    min_index=countlist.index(min_value)
    min_class=classlist[min_index]
    print(max_class, ' has the most images= ',max_value, ' ', min_class, ' has the least images= ', min_value)
    # lets get the average height and width of a sample of the train images
    ht=0
    wt=0
    # select 100 random samples of train_df
    train_df_sample=train_df.sample(n=100, random_state=123,axis=0)
    for i in range (len(train_df_sample)):
        fpath=train_df_sample['filepaths'].iloc[i]
        img=plt.imread(fpath)
        shape=img.shape
        ht += shape[0]
        wt += shape[1]
    print('average height= ', ht//100, ' average width= ', wt//100, 'aspect ratio= ', ht/wt)
    return train_df, test_df, valid_df, classes
train_df, test_df, valid_df, classes = make_df(save_path)
class_count=len(classes)

train_df lenght:  6203   test_df length:  776   valid_df length:  775
The number of classes in the dataset is:  5
            CLASS               IMAGE COUNT 
     Alternaria Leaf Spot          1206     
      Bacterial Soft Rot           1121     
          Black Rot                 627     
         Downy Mildew               721     
           Healthy                 2528     
Healthy  has the most images=  2528   Black Rot  has the least images=  627
average height=  300  average width=  300 aspect ratio=  1.0


In [4]:
def balance(df, n, working_dir, img_size):
    df=df.copy()
    print('Initial length of dataframe is ', len(df))
    aug_dir=os.path.join(working_dir, 'aug')# directory to store augmented images
    if os.path.isdir(aug_dir):# start with an empty directory
        shutil.rmtree(aug_dir)
    os.mkdir(aug_dir)
    for label in df['labels'].unique():
        dir_path=os.path.join(aug_dir,label)
        os.mkdir(dir_path) # make class directories within aug directory
    # create and store the augmented images
    total=0
    gen=ImageDataGenerator(horizontal_flip=True,  rotation_range=20, width_shift_range=.2,
                                  height_shift_range=.2, zoom_range=.2)
    groups=df.groupby('labels') # group by class
    for label in df['labels'].unique():  # for every class
        group=groups.get_group(label)  # a dataframe holding only rows with the specified label
        sample_count=len(group)   # determine how many samples there are in this class
        if sample_count< n: # if the class has less than target number of images
            aug_img_count=0
            delta=n - sample_count  # number of augmented images to create
            target_dir=os.path.join(aug_dir, label)  # define where to write the images
            msg='{0:40s} for class {1:^30s} creating {2:^5s} augmented images'.format(' ', label, str(delta))
            print(msg, '\r', end='') # prints over on the same line
            aug_gen=gen.flow_from_dataframe( group,  x_col='filepaths', y_col=None, target_size=img_size,
                                            class_mode=None, batch_size=1, shuffle=False,
                                            save_to_dir=target_dir, save_prefix='aug-', color_mode='rgb',
                                            save_format='jpg')
            while aug_img_count<delta:
                images=next(aug_gen)
                aug_img_count += len(images)
            total +=aug_img_count
    print('Total Augmented images created= ', total)
    # create aug_df and merge with train_df to create composite training set ndf
    aug_fpaths=[]
    aug_labels=[]
    classlist=os.listdir(aug_dir)
    for klass in classlist:
        classpath=os.path.join(aug_dir, klass)
        flist=os.listdir(classpath)
        for f in flist:
            fpath=os.path.join(classpath,f)
            aug_fpaths.append(fpath)
            aug_labels.append(klass)
    Fseries=pd.Series(aug_fpaths, name='filepaths')
    Lseries=pd.Series(aug_labels, name='labels')
    aug_df=pd.concat([Fseries, Lseries], axis=1)

    # save augmented dataframe to CSV
    try:
        aug_csv = os.path.join(working_dir, 'augmented_filepaths.csv')
        aug_df.to_csv(aug_csv, index=False)
    except Exception as _e:
        print('Warning: could not save aug_df to CSV:', _e)

    df=pd.concat([df,aug_df], axis=0).reset_index(drop=True)

    # save merged dataframe (original + augmented) to CSV
    try:
        merged_csv = os.path.join(working_dir, 'train_augmented.csv')
        df.to_csv(merged_csv, index=False)
    except Exception as _e:
        print('Warning: could not save merged df to CSV:', _e)

    print('Length of augmented dataframe is now ', len(df))
    try:
        print(f"Saved {len(aug_df)} augmented rows to {aug_csv} and merged dataset to {merged_csv}")
    except NameError:
        pass
    return df

n=200 # number of samples in each class
working_dir=r'./' # directory to store augmented images
img_size=(300,300) # size of augmented images
train_df=balance(train_df, n, working_dir, img_size)

Initial length of dataframe is  6203
Total Augmented images created=  0
Length of augmented dataframe is now  6203
Saved 0 augmented rows to ./augmented_filepaths.csv and merged dataset to ./train_augmented.csv


In [5]:
# Repair corrupted images (run after scan_folder produced bad_files.csv)
from PIL import Image
import shutil, csv

def repair_and_quarantine(bad_csv, quarantine_dir, reencode_quality=95):
    os.makedirs(quarantine_dir, exist_ok=True)
    bad_paths = []
    with open(bad_csv, 'r') as f:
        reader = csv.DictReader(f)
        for r in reader:
            bad_paths.append(r['path'])

    moved = []
    for p in bad_paths:
        try:
            img = Image.open(p).convert('RGB')
            img.save(p, format='JPEG', quality=reencode_quality)  # re-encode in place
        except Exception:
            dst = os.path.join(quarantine_dir, os.path.basename(p))
            shutil.move(p, dst)
            moved.append((p, dst))

    print(f"Attempted repair on {len(bad_paths)} files — quarantined {len(moved)} irrecoverable files")
    return moved

# usage
quarantine_dir = os.path.join(working_dir, 'quarantine')
bad_csv = os.path.join(working_dir, 'bad_files.csv')  # or wherever scan_folder wrote it
if os.path.exists(bad_csv):
    repair_and_quarantine(bad_csv, quarantine_dir)
else:
    print('bad_files.csv not found; run scan_folder first')

bad_files.csv not found; run scan_folder first


In [6]:
# Scan for corrupted images and write bad_files.csv
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import os, csv

def scan_folder(root, out_csv):
    bad = []
    for dirpath, _, files in os.walk(root):
        for fn in files:
            path = os.path.join(dirpath, fn)
            try:
                with Image.open(path) as im:
                    im.verify()
                # re-open to fully decode
                with Image.open(path) as im:
                    im.load()
            except Exception as e:
                bad.append((path, repr(e)))
    with open(out_csv, 'w', newline='') as wf:
        writer = csv.writer(wf)
        writer.writerow(['path', 'error'])
        writer.writerows(bad)
    print(f"Wrote {len(bad)} bad paths to {out_csv}")
    return bad

# run scan on the resized dataset
bad_csv = os.path.join(working_dir, 'bad_files.csv')
root = os.path.join(working_dir, 'cali')
if os.path.isdir(root):
    scan_folder(root, out_csv=bad_csv)
else:
    print(f'Resized dataset not found at {root}; run preprocessing in augmented.ipynb')


Resized dataset not found at ./cali; run preprocessing in augmented.ipynb


In [7]:
# Rebuild cleaned CSVs (run after repairing/quarantining)
def write_clean_csv(input_csv, out_csv, quarantine_dir):
    df = pd.read_csv(input_csv)
    # drop any rows whose files were moved to quarantine
    if os.path.isdir(quarantine_dir):
        quarantined = set(os.listdir(quarantine_dir))
        df = df[~df['filepaths'].apply(lambda p: os.path.basename(p) in quarantined)].reset_index(drop=True)
    df.to_csv(out_csv, index=False)
    print('Wrote cleaned CSV:', out_csv)

write_clean_csv(os.path.join(working_dir,'train_augmented.csv'),
                os.path.join(working_dir,'train_augmented_clean.csv'),
                quarantine_dir)

Wrote cleaned CSV: ./train_augmented_clean.csv
